# 💳 Task 1: Credit Scoring Model
**CodeAlpha Machine Learning Internship**

Predict creditworthiness using classification algorithms.

> ✅ **Runtime:** Runtime → Change runtime type → **GPU** (optional for this task)

In [ ]:
# ── Install & Import ──────────────────────────
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","scikit-learn","pandas","numpy","matplotlib","seaborn"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score,
                              accuracy_score, precision_score, recall_score)
from sklearn.pipeline import Pipeline

print("✅ Libraries imported successfully!")

In [ ]:
# ── Load Dataset ─────────────────────────────
print("Loading German Credit Dataset...")
credit = fetch_openml('credit-g', version=1, as_frame=True)
df = credit.frame
df['class'] = (df['class'] == 'good').astype(int)

print(f"✅ Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['class'].value_counts().rename({1:'Good Credit',0:'Bad Credit'}))
df.head()

In [ ]:
# ── Exploratory Data Analysis ────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Credit Scoring — EDA Dashboard', fontsize=16, fontweight='bold')

axes[0,0].pie(df['class'].value_counts(),
              labels=['Good Credit','Bad Credit'],
              colors=['#2ecc71','#e74c3c'],
              autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('Credit Class Distribution')

axes[0,1].hist(df['age'].astype(float), bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0,1].set_title('Age Distribution'); axes[0,1].set_xlabel('Age')

axes[0,2].hist(df['credit_amount'].astype(float), bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[0,2].set_title('Credit Amount Distribution')

axes[1,0].hist(df['duration'].astype(float), bins=20, color='#e67e22', edgecolor='white', alpha=0.8)
axes[1,0].set_title('Loan Duration Distribution')

good = df[df['class']==1]['credit_amount'].astype(float)
bad  = df[df['class']==0]['credit_amount'].astype(float)
axes[1,1].boxplot([good, bad], labels=['Good','Bad'], patch_artist=True,
                  boxprops=dict(facecolor='#3498db', alpha=0.6))
axes[1,1].set_title('Credit Amount by Class')

axes[1,2].text(0.5,0.5,'No Missing Values ✅',ha='center',va='center',
               fontsize=14,color='green',transform=axes[1,2].transAxes)
axes[1,2].set_title('Missing Values Check')

plt.tight_layout()
plt.savefig('task1_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA complete!")

In [ ]:
# ── Preprocessing ────────────────────────────
df_enc = df.copy()
for col in df_enc.select_dtypes(include='category').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

X = df_enc.drop('class', axis=1)
y = df_enc['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# ── Train All Models ─────────────────────────
models = {
    "Logistic Regression": Pipeline([('s',StandardScaler()),('c',LogisticRegression(max_iter=1000,random_state=42))]),
    "Decision Tree":       Pipeline([('s',StandardScaler()),('c',DecisionTreeClassifier(max_depth=6,random_state=42))]),
    "Random Forest":       Pipeline([('s',StandardScaler()),('c',RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1))]),
    "Gradient Boosting":   Pipeline([('s',StandardScaler()),('c',GradientBoostingClassifier(n_estimators=200,learning_rate=0.05,random_state=42))])
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    results[name] = {
        'accuracy': accuracy_score(y_test,y_pred),
        'f1':       f1_score(y_test,y_pred),
        'roc_auc':  roc_auc_score(y_test,y_proba),
        'cv_auc':   cv_scores.mean(),
        'y_pred':   y_pred, 'y_proba': y_proba, 'model': model
    }
    print(f"✅ {name:<22} | AUC: {results[name]['roc_auc']:.4f} | F1: {results[name]['f1']:.4f}")

In [ ]:
# ── Results Dashboard ────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Credit Scoring — Results Dashboard', fontsize=16, fontweight='bold')
colors = ['#3498db','#2ecc71','#e74c3c','#9b59b6']
model_names = list(results.keys())

# Metric comparison
metrics = ['accuracy','f1','roc_auc']
x = np.arange(len(metrics)); w = 0.2
for i,(name,res) in enumerate(results.items()):
    axes[0,0].bar(x+i*w,[res[m] for m in metrics],w,label=name,color=colors[i],alpha=0.85)
axes[0,0].set_xticks(x+w*1.5); axes[0,0].set_xticklabels(['Accuracy','F1','ROC-AUC'])
axes[0,0].set_title('Model Comparison'); axes[0,0].legend(fontsize=7); axes[0,0].set_ylim(0.5,1.0)

# ROC Curves
for i,(name,res) in enumerate(results.items()):
    fpr,tpr,_ = roc_curve(y_test,res['y_proba'])
    axes[0,1].plot(fpr,tpr,color=colors[i],lw=2,label=f"{name} ({res['roc_auc']:.3f})")
axes[0,1].plot([0,1],[0,1],'k--'); axes[0,1].set_title('ROC Curves')
axes[0,1].legend(fontsize=7); axes[0,1].set_xlabel('FPR'); axes[0,1].set_ylabel('TPR')

# Confusion Matrix - Best model
best = max(results, key=lambda k: results[k]['roc_auc'])
cm = confusion_matrix(y_test, results[best]['y_pred'])
sns.heatmap(cm,annot=True,fmt='d',ax=axes[0,2],cmap='Blues',
            xticklabels=['Bad','Good'],yticklabels=['Bad','Good'])
axes[0,2].set_title(f'Confusion Matrix — {best}')

# CV AUC
cv_means = [results[n]['cv_auc'] for n in model_names]
bars = axes[1,0].bar(model_names, cv_means, color=colors, alpha=0.85)
axes[1,0].set_title('Cross-Validation AUC (5-Fold)'); axes[1,0].set_ylim(0.6,0.9)
axes[1,0].tick_params(axis='x',rotation=15)
for bar,val in zip(bars,cv_means):
    axes[1,0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,f'{val:.3f}',ha='center',fontsize=9)

# Feature Importance
rf = results['Random Forest']['model'].named_steps['c']
imp = pd.Series(rf.feature_importances_,index=X.columns).sort_values(ascending=False)[:12]
imp.plot(kind='barh',ax=axes[1,1],color='#3498db',alpha=0.8)
axes[1,1].set_title('Top 12 Feature Importances (Random Forest)'); axes[1,1].invert_yaxis()

# Summary
axes[1,2].axis('off')
summary = "\n".join([f"  {n:<22} {r['roc_auc']:.4f}" for n,r in results.items()])
axes[1,2].text(0.1,0.9,f"RESULTS SUMMARY\n{'─'*35}\n{summary}\n\n✅ Best: {best}",
               transform=axes[1,2].transAxes,va='top',family='monospace',fontsize=10)

plt.tight_layout()
plt.savefig('task1_results.png',dpi=150,bbox_inches='tight')
plt.show()

print("\n" + "="*55)
print(f"{'Model':<25} {'Accuracy':>9} {'F1':>7} {'AUC':>7}")
print("-"*55)
for name,res in results.items():
    mark = " ← Best" if name==best else ""
    print(f"{name:<25} {res['accuracy']:>9.4f} {res['f1']:>7.4f} {res['roc_auc']:>7.4f}{mark}")
print("="*55)

In [ ]:
# ── Download Output Files ────────────────────
from google.colab import files
files.download('task1_eda.png')
files.download('task1_results.png')
print("✅ Task 1 Complete! Files downloaded.")